# Robust Susceptibility Validation Input Generator

`validation_hist.ipynb` と同じ validation 用の摂動点を、HFSS 実行前の `parametric_input.csv` として作成する notebook です。

## 使い方

1. `x_center = []` に 13 次元の中心パラメータを入力します。
2. `N_VALIDATION = 100`, `PERTURBATION_STD_RATIO = 0.01`, `RANDOM_SEED = 101` の条件で乱数を生成します。
3. index `0` を中心点、index `1`〜`100` を摂動点とする `101 x 13` の DataFrame を作ります。
4. `_config.toml` の `[hfss]` / `param_groups` から取得した parameter names を columns にします。
5. timestamp 付き run directory を作成し、その中に `parametric_input.csv` を保存します。


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import lib_config as config
import lib_backbone
import lib_gp

%load_ext autoreload
%autoreload 2


In [ ]:
# Load parameter metadata from _config.toml.
_config = config._loadConfig(Path("./_config.toml"))
app_config = config.initParams(_config, debug=True)

backbone = lib_backbone.Backbone(config=app_config)
cfg = app_config.hfss
ROUND_DECIMALS = app_config.runtime.round_decimals

LOWER_BOUNDS = np.asarray(cfg.lower_bounds, dtype=float)
UPPER_BOUNDS = np.asarray(cfg.upper_bounds, dtype=float)
BOUNDS = np.vstack([LOWER_BOUNDS, UPPER_BOUNDS]).T
PARAM_NAMES = list(cfg.param_names)

print("n_params:", cfg.n_params)
print("PARAM_NAMES:", PARAM_NAMES)
print("ROUND_DECIMALS:", ROUND_DECIMALS)


In [ ]:
# ===== User input =====
# cfg.n_params (= 13) と同じ長さの中心パラメータをここに入力してください。
# Example:
# x_center = np.asarray([
#     6.225785,
#     4.065235,
#     9.188137,
#     9.788713,
#     1.000000,
#     1.071733,
#     2.000000,
#     1.456739,
#     1.835538,
#     0.537034,
#     2.000000,
#     3.711260,
#     6.000000,
# ], dtype=float)
x_center = []

N_VALIDATION = 100
PERTURBATION_STD_RATIO = 0.01
RANDOM_SEED = 101


In [ ]:
def validate_x_center(x, n_params, lower_bounds, upper_bounds):
    """Validate and round the user-provided center vector."""
    x = np.asarray(x, dtype=float).reshape(-1)
    if x.size != n_params:
        raise ValueError(
            f"x_center must have length cfg.n_params={n_params}, but got length {x.size}. "
            "Set x_center before generating parametric_input.csv."
        )

    below = x < lower_bounds
    above = x > upper_bounds
    if np.any(below) or np.any(above):
        bad = [
            (PARAM_NAMES[i], float(x[i]), float(lower_bounds[i]), float(upper_bounds[i]))
            for i in np.where(below | above)[0]
        ]
        raise ValueError(f"x_center has out-of-bounds values: {bad}")

    return np.round(x, ROUND_DECIMALS)


def build_validation_sigma():
    """Build diagonal covariance from PERTURBATION_STD_RATIO and parameter ranges."""
    bounds_width = UPPER_BOUNDS - LOWER_BOUNDS
    perturbation_std = PERTURBATION_STD_RATIO * bounds_width
    sigma = np.diag(perturbation_std ** 2)

    expected_shape = (cfg.n_params, cfg.n_params)
    if sigma.shape != expected_shape:
        raise ValueError(f"Sigma must have shape {expected_shape}, got {sigma.shape}.")
    return sigma


In [ ]:
x_center = validate_x_center(x_center, cfg.n_params, LOWER_BOUNDS, UPPER_BOUNDS)
VALIDATION_SIGMA = build_validation_sigma()
rng = np.random.default_rng(RANDOM_SEED)

# Generate only perturbation rows here. The center row is prepended as index 0 below.
Z_perturbation = lib_gp.sample_input_perturbations(
    x=x_center,
    Sigma=VALIDATION_SIGMA,
    n_samples=N_VALIDATION,
    bounds=BOUNDS,
    rng=rng,
)
Z_perturbation = np.round(Z_perturbation, ROUND_DECIMALS)

# index 0: center, index 1..N_VALIDATION: perturbations
Z_validation_with_center = np.vstack([x_center, Z_perturbation])
parametric_input_df = pd.DataFrame(Z_validation_with_center, columns=PARAM_NAMES)
parametric_input_df.index.name = "sample_idx"

print("DataFrame shape:", parametric_input_df.shape)
print("Center row index:", parametric_input_df.index[0])
print("Perturbation index range:", f"{parametric_input_df.index[1]}..{parametric_input_df.index[-1]}")
display(parametric_input_df.head())


In [ ]:
# Create a timestamped run directory using the same Backbone directory convention as the other notebooks.
backbone.mkdir()
run_dir = backbone._get_dir_run()

parametric_input_csv = run_dir / "parametric_input.csv"
parametric_input_df.to_csv(parametric_input_csv, index=False)

print(f"Saved parametric input CSV: {parametric_input_csv}")
print(f"Rows x columns: {parametric_input_df.shape[0]} x {parametric_input_df.shape[1]}")


In [ ]:
# Optional: inspect the saved CSV.
saved_parametric_input_df = pd.read_csv(parametric_input_csv)
display(saved_parametric_input_df.head())
